In [6]:
'''
group data
'''
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_excel('C:/Users/fu/Downloads/Dokumente/Tariffmodelle/TG/2024_prices_Industrie_ohne_Netzentgelt.xlsx', index_col=[0])
prices = df[['Wholesale price corrected']].copy()
prices.rename(columns={"Wholesale price corrected": "tariff"}, inplace=True)
# prices = prices[prices.index <= '2024-09-30 23:00:00']

In [2]:
prices

,tariff
Time,
2024-01-01 00:00:00,2.75050
2024-01-01 01:00:00,2.74105
2024-01-01 02:00:00,2.74000
2024-01-01 03:00:00,2.73905
2024-01-01 04:00:00,2.73715
...,...
2024-12-31 19:00:00,9.85585
2024-12-31 20:00:00,6.47380
2024-12-31 21:00:00,4.38850


In [3]:
'''
to resample and save file for Model input
'''
prices = prices.resample("15T").ffill()
prices['tariff'] = prices['tariff'] * 0.01
prices.to_csv(r"C:\Users\fu\Downloads\Dokumente\PISA\focus-framework-main\focus-framework-main\input_files\data\prices\dynamisch_Endkunden_Industrie_1.24_Netzentgelt.csv")

C:\Users\fu\AppData\Local\Temp\ipykernel_28584\2250977248.py:4: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  prices = prices.resample("15T").ffill()


In [7]:
import holidays

# to group holidays from now to 2025
# start_time = '2024-12-01 00:00'
# end_time = '2025-12-31 23:00'
# time_index = pd.date_range(start=start_time, end=end_time, freq='H')
# df = pd.DataFrame(index=time_index)
# prices = df


de_holidays = holidays.Germany(prov='BW')

def classify_weekday(date):
    if date.month in [4, 5, 6, 7, 8, 9, 10]:
        return 'summer weekday'
    elif date.month in [1, 2, 3, 11, 12]:
        return 'winter weekday'

def classify_weekend(date):
    if date.month in [4, 5, 6, 7, 8, 9, 10]:
        return 'summer weekend'
    elif date.month in [1, 2, 3, 11, 12]: 
        return 'winter weekend'

def classify_season(date): # keine Trennung Wochenende für Industrie
    if date.month in [4, 5, 6, 7, 8, 9, 10]:
        return 'summer'
    elif date.month in [1, 2, 3, 11, 12]:
        return 'winter'

def days_grouping(date):
    if date.weekday() < 5:
        prices.loc[date, 'group'] = classify_weekday(date)
    if date.weekday() > 4:
        prices.loc[date, 'group'] = classify_weekend(date)
        
    # Feiertage und Brückentage:
    if date in de_holidays:
        prices.loc[date, 'group'] = classify_weekend(date)
    if date.weekday()==0 and (date + pd.Timedelta(days=1) in de_holidays):
        prices.loc[date, 'group'] = classify_weekend(date)
        #print(date)
    if date.weekday()==4 and (date - pd.Timedelta(days=1) in de_holidays):
        prices.loc[date, 'group'] = classify_weekend(date)
        #print(date)

    # Neujahr_2023 = pd.date_range(start=pd.Timestamp('2023-01-02 00:00:00'), end=pd.Timestamp('2023-01-06 23:00:00'), freq='H')
    range1 = pd.date_range(start="2024-12-25 00:00:00", end="2024-12-31 23:00:00", freq="H")
    range2 = pd.date_range(start="2024-01-01 00:00:00", end="2024-01-06 23:00:00", freq="H")
    Neujahr_2024 = range1.union(range2)
    if date in Neujahr_2024:
        prices.loc[date, 'group'] = classify_weekend(date)

def days_grouping_industry(date):
    prices.loc[date, 'group'] = classify_season(date)

for date in prices.index.to_series():
    days_grouping_industry(date)
    
prices

,tariff,group
Time,,
2024-01-01 00:00:00,2.75050,winter
2024-01-01 01:00:00,2.74105,winter
2024-01-01 02:00:00,2.74000,winter
2024-01-01 03:00:00,2.73905,winter
2024-01-01 04:00:00,2.73715,winter
...,...,...
2024-12-31 19:00:00,9.85585,winter
2024-12-31 20:00:00,6.47380,winter
2024-12-31 21:00:00,4.38850,winter


In [8]:
prices.to_csv(r'C:\Users\fu\Downloads\Dokumente\Tariffmodelle\TG\2024_data_grouped_Industrie_1.24_Netzentgelt.csv')

In [4]:
for date, name in sorted(de_holidays.items()):
    print(date, name)

2024-01-01 Neujahr
2024-01-06 Heilige Drei Könige
2024-03-29 Karfreitag
2024-04-01 Ostermontag
2024-05-01 Erster Mai
2024-05-09 Christi Himmelfahrt
2024-05-20 Pfingstmontag
2024-05-30 Fronleichnam
2024-10-03 Tag der Deutschen Einheit
2024-11-01 Allerheiligen
2024-12-25 Erster Weihnachtstag
2024-12-26 Zweiter Weihnachtstag
